# Tarea 2 · Parte 1 — Voces y documentos peruanos: del archivo escaneado al texto buscable

**Pregunta guía:** ¿Podemos convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

| Dato | Valor |
|---|---|
| Fecha de extracción | _(completar con la salida de la celda de versiones)_ |
| Entorno | `uv sync --group docai --group audio` (repositorio de la clase) |
| PaddleOCR | _(versión impresa abajo)_ — PP-OCR en español, `lang="es"` |
| ASR | faster-whisper large-v3-turbo (int8, CPU, VAD Silero activado) vía `voz.py` de la Sesión 13 |
| Diarización | huellas WeSpeaker ResNet34 (pyannote) + clustering jerárquico (`voz.diarizar`) |

**Fuentes y licencias** (detalle por archivo en `datos/fuentes.csv`):

- **Documentos:** Wikimedia Commons (dominio público, CC0, CC BY, CC BY-SA; autor y licencia por archivo) y capturas propias sin datos personales (CC BY 4.0).
- **Audio:** entrevistas de Wikimedia Commons (CC BY-SA 4.0), noticiarios de la Biblioteca Nacional del Perú en Commons (dominio público) y frases leídas de OpenSLR 73 *Crowdsourced high-quality Peruvian Spanish speech data set* (Google, CC BY-SA 4.0).

Los archivos originales quedan en `datos/raw/` (fuera de Git) y se recuperan con `uv run python descargas.py bajar`.

In [1]:
import os, json, time, warnings
from datetime import date
from pathlib import Path

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import soundfile as sf

import paddleocr, faster_whisper
import herramientas as H
import voz

DATOS = Path("datos")
RAW_DOC, RAW_AUD = DATOS / "raw" / "documentos", DATOS / "raw" / "audio"
FECHA = date.today().isoformat()
pd.set_option("display.max_colwidth", 120)

print(f"Fecha de extracción: {FECHA}")
print(f"PaddleOCR {paddleocr.__version__} · faster-whisper {faster_whisper.__version__}")

Fecha de extracción: 2026-10-25
PaddleOCR 3.7.0 · faster-whisper 1.2.1


## 1 · Corpus y fuentes

`descargas.py bajar` dejó los archivos en `datos/raw/` y su atribución en `datos/fuentes_crudas.csv`.
Primero se eliminan los duplicados exactos (mismo hash SHA-256).

In [2]:
fuentes = pd.read_csv(DATOS / "fuentes_crudas.csv")
docs = fuentes[fuentes["modalidad"] == "documento"].reset_index(drop=True)
docs = H.marcar_duplicados(docs, "archivo")
descartes = docs.loc[docs["duplicado"], ["id", "archivo"]].assign(motivo="duplicado (mismo hash)")
docs = docs[~docs["duplicado"]].reset_index(drop=True)

print(f"Documentos únicos: {len(docs)} · duplicados descartados: {len(descartes)}")
docs.groupby("tipo").size().rename("n").to_frame()

Documentos únicos: 25 · duplicados descartados: 0


,n
tipo,
foto,8
impreso,8
manuscrito,9


## 2 · OCR con PaddleOCR

- **Idioma:** `lang="es"`.
- **Preprocesamiento:** corrección de rotación EXIF, RGB y lado máximo de 1600 px (copias en `datos/raw/documentos/prep/`).
- **Orientación por línea activada** (`use_textline_orientation=True`) para las fotos inclinadas; sin desdoblado de página.
- **Respaldo crudo:** cada resultado se guarda como JSON por línea (texto, confianza, caja) en `datos/raw/documentos/ocr_json/`, y las 3 líneas de menor confianza como recortes en `datos/raw/documentos/recortes/`.

In [3]:
ocr = H.crear_ocr()
MODELO_OCR = f"PaddleOCR {paddleocr.__version__} (lang=es, textline_orientation=on, sin unwarping, lado máx. 1600 px)"
(RAW_DOC / "ocr_json").mkdir(parents=True, exist_ok=True)

filas = []
for d in docs.itertuples():
    prep = H.preprocesar(Path(d.archivo), RAW_DOC / "prep" / f"{d.id}.jpg")
    cache = RAW_DOC / "ocr_json" / f"{d.id}.json"
    t0 = time.perf_counter()
    if cache.exists():                      # respaldo crudo de una corrida anterior: no se repite el OCR
        lineas = pd.read_json(cache, orient="records")
    else:
        lineas = H.leer_ocr(ocr, prep)
        lineas.to_json(cache, orient="records", force_ascii=False)
        H.recortes(lineas, prep, RAW_DOC / "recortes")
    seg = time.perf_counter() - t0
    filas.append({"id": d.id, "texto_ocr": H.unir_lineas(lineas), "n_lineas": len(lineas),
                  "confianza_media": lineas["confianza"].mean() if len(lineas) else 0.0, "segundos": round(seg, 2)})

ocr_res = docs.merge(pd.DataFrame(filas), on="id")
ocr_res[["id", "tipo", "n_lineas", "confianza_media", "segundos"]].round(3)

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/valeria/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/valeria/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/valeria/.paddlex/official_models/PP-OCRv6_medium_rec`.


,id,tipo,n_lineas,confianza_media,segundos
0,doc_001,impreso,28,0.989,0.0
1,doc_002,impreso,44,0.942,0.0
2,doc_003,impreso,34,0.855,0.0
3,doc_004,impreso,10,0.771,0.0
4,doc_005,impreso,5,0.958,0.0
5,doc_006,impreso,154,0.665,0.0
6,doc_007,impreso,231,0.982,0.0
7,doc_008,impreso,124,0.957,0.0
8,doc_009,foto,6,0.930,0.0
9,doc_010,foto,22,0.936,0.0


### Documentos ilegibles

Criterio: un documento es ilegible si una persona no puede leerlo. Las 25 imágenes se revisaron a ojo y todas son legibles, así que no se descarta ninguno. Los documentos donde el OCR lee menos de 20 caracteres se conservan, porque son justamente los casos difíciles.

In [4]:
# Ilegible = una persona no puede leer el documento. Las 25 imágenes se revisaron a ojo al armar el corpus y
# todas son legibles. Que el OCR lea poco NO es motivo de descarte: es un caso difícil que se evalúa en la Parte 2.
ILEGIBLES_HUMANO = []
ilegible = ocr_res["id"].isin(ILEGIBLES_HUMANO)
descartes = pd.concat([descartes, ocr_res.loc[ilegible, ["id", "archivo"]].assign(motivo="ilegible para una persona")])
ocr_res = ocr_res[~ilegible].reset_index(drop=True)

poco_texto = ocr_res.loc[ocr_res["texto_ocr"].str.len() < 20, "id"].tolist()
print(f"Ilegibles descartados: {ilegible.sum()} · OCR con menos de 20 caracteres (se conservan como casos difíciles): {poco_texto}")
cuenta = ocr_res["tipo"].value_counts().reindex(["impreso", "foto", "manuscrito"]).fillna(0).astype(int)
minimos = pd.Series({"impreso": 8, "foto": 8, "manuscrito": 9})
pd.DataFrame({"tenemos": cuenta, "mínimo": minimos, "ok": cuenta >= minimos})

Ilegibles descartados: 0 · OCR con menos de 20 caracteres (se conservan como casos difíciles): ['doc_011']


,tenemos,mínimo,ok
impreso,8,8,True
foto,8,8,True
manuscrito,9,9,True


## 3 · ASR: clips, VAD, transcripción y diarización

1. Las entrevistas largas se cortan en **clips de ~45 segundos** siempre en una pausa (VAD Silero), para tener unidades comparables.
2. Las frases de OpenSLR 73 se usan como clips cortos de habla leída (un hablante).
3. Los clips con menos de 10 % de voz se descartan como «sin voz».
4. Transcripción con faster-whisper (`vad_filter=True`, que evita que Whisper alucine en silencios y música).
5. **Diarización** en entrevistas y noticiarios con `voz.diarizar` (número de hablantes estimado con umbral coseno 0.7).
6. Exportación de cada clip a **SRT** y **TXT** en `datos/raw/audio/salidas/`.

In [5]:
CLIPS = RAW_AUD / "clips"; CLIPS.mkdir(parents=True, exist_ok=True)
# Se excluyen las dos grabaciones más largas (discurso de 13 min y entrevista de 9 min) para acotar
# el tiempo de cómputo en CPU: aun así quedan ~49 min de entrevistas y noticiarios + OpenSLR.
EXCLUIR = {"fuente_13", "fuente_16"}
fuentes = fuentes[~fuentes["id"].isin(EXCLUIR)]
fuentes_aud = fuentes[fuentes["modalidad"] == "audio"]

clips = []
for f in fuentes_aud.itertuples():
    audio = voz.cargar_audio(f.archivo)
    for k, (a, b) in enumerate(H.cortar_en_clips(audio)):
        ruta = CLIPS / f"{f.id}_{k:02d}.wav"
        sf.write(ruta, audio[int(a * voz.SR):int(b * voz.SR)], voz.SR)
        clips.append({"fuente_id": f.id, "tipo_audio": f.tipo, "archivo_audio": str(ruta),
                      "inicio_en_fuente": round(a, 1), "duracion_seg": round(b - a, 1)})

# frases leídas de OpenSLR 73 (descargadas por descargas.py en datos/raw/audio/es_peru/)
openslr = pd.read_csv(RAW_AUD / "es_peru" / "transcripciones.csv")
for r in openslr.itertuples():
    ruta = RAW_AUD / "es_peru" / r.archivo
    clips.append({"fuente_id": "openslr73", "tipo_audio": "habla leída", "archivo_audio": str(ruta),
                  "inicio_en_fuente": 0.0, "duracion_seg": round(sf.info(ruta).duration, 1)})

clips = pd.DataFrame(clips)
clips = H.marcar_duplicados(clips, "archivo_audio")
clips["prop_voz"] = [H.proporcion_voz(voz.cargar_audio(r)) for r in clips["archivo_audio"]]

sin_voz = clips["prop_voz"] < 0.10
descartes = pd.concat([descartes,
    clips.loc[clips["duplicado"], ["archivo_audio"]].assign(motivo="audio duplicado (mismo hash)"),
    clips.loc[sin_voz & ~clips["duplicado"], ["archivo_audio"]].assign(motivo="sin voz (<10 % de habla según VAD)")])
clips = clips[~clips["duplicado"] & ~sin_voz].reset_index(drop=True)
clips.insert(0, "id", [f"clip_{i:03d}" for i in range(1, len(clips) + 1)])

minutos = clips["duracion_seg"].sum() / 60
print(f"{len(clips)} clips · {minutos:.1f} minutos · descartados sin voz: {sin_voz.sum()}")
assert len(clips) >= 20 and minutos >= 30, "Faltan clips o minutos: agrega audios a seleccion_audio.csv"

97 clips · 51.6 minutos · descartados sin voz: 2


In [6]:
t = voz.Transcriptor("faster-whisper")
MODELO_ASR = "faster-whisper large-v3-turbo (int8, CPU, VAD on, beam 5)"
SALIDAS = RAW_AUD / "salidas"; SALIDAS.mkdir(exist_ok=True)
DIARIZAR = {"entrevista", "noticiario"}

filas = []
for c in clips.itertuples():
    cache = SALIDAS / f"{c.id}.json"
    if cache.exists():                      # corrida anterior del mismo clip: se reutiliza
        guardado = json.loads(cache.read_text(encoding="utf-8"))
        segs, seg_proc = [voz.Segmento(**s) for s in guardado["segmentos"]], guardado["segundos"]
    else:
        audio = voz.cargar_audio(c.archivo_audio)
        t0 = time.perf_counter()
        segs = t.segmentos(audio, idioma="es")
        if c.tipo_audio in DIARIZAR and len(segs) > 1:
            segs = voz.diarizar(audio, segs)
        seg_proc = time.perf_counter() - t0
        cache.write_text(json.dumps({"archivo": c.archivo_audio, "segundos": seg_proc,
                                     "segmentos": [vars(s) for s in segs]}, ensure_ascii=False), encoding="utf-8")
    n_h = max(1, len({s.hablante for s in segs if s.hablante}))
    (SALIDAS / f"{c.id}.srt").write_text(voz.a_srt(segs), encoding="utf-8")
    (SALIDAS / f"{c.id}.txt").write_text(voz.a_texto(segs), encoding="utf-8")
    filas.append({"id": c.id, "texto_asr": H.texto_con_hablantes(segs) if n_h > 1 else " ".join(s.texto for s in segs),
                  "n_hablantes": n_h, "segundos_proceso": round(seg_proc, 1)})

asr_res = clips.merge(pd.DataFrame(filas), on="id")
asr_res["rtf"] = (asr_res["segundos_proceso"] / asr_res["duracion_seg"]).round(2)
print(f"Factor de tiempo real medio: {asr_res['rtf'].mean():.2f}")
print(f"Clips con 2+ hablantes: {(asr_res['n_hablantes'] >= 2).sum()}")
asr_res[["id", "tipo_audio", "duracion_seg", "n_hablantes", "rtf"]].head(10)

Factor de tiempo real medio: 0.56
Clips con 2+ hablantes: 33


,id,tipo_audio,duracion_seg,n_hablantes,rtf
0,clip_001,entrevista,46.3,2,0.64
1,clip_002,entrevista,45.5,2,0.33
2,clip_003,entrevista,44.5,2,0.28
3,clip_004,entrevista,47.9,3,0.26
4,clip_005,entrevista,47.2,1,0.25
5,clip_006,entrevista,47.4,3,0.26
6,clip_007,entrevista,46.6,2,0.23
7,clip_008,entrevista,48.0,1,0.27
8,clip_009,entrevista,46.0,2,0.27
9,clip_010,entrevista,50.9,2,0.25


### Revisión manual contra alucinaciones

Se escuchan 5 clips al azar (semilla fija) y se comparan con su texto. Anotar aquí lo encontrado
(frases repetidas, texto inventado en la música de los noticiarios, etc.):

- _(completar después de escuchar)_

In [7]:
from IPython.display import Audio, display
for r in asr_res.sample(5, random_state=7).itertuples():
    print(f"── {r.id} · {r.tipo_audio} · {r.duracion_seg:.0f} s\n{r.texto_asr[:400]}")
    display(Audio(r.archivo_audio))

── clip_021 · entrevista · 46 s
Esta afrenda. Hoy decimos a los descendientes de estos bárbaros que se camuflan en el Congreso que no pasará el intento de cambiar de nombre el Ministerio de la Mujer, porque en 500 años de exclusión enseñamos a nuestros hijos a luchar con dignidad para construir una patria de igualdad y libertad. Somos peruanos y no regresaremos al mundo de la cavernícola de tanto extrañan estos herederos del col


── clip_072 · habla leída · 5 s
Es muy fácil reguardar su hogar de los huracanes.


── clip_077 · habla leída · 8 s
Diego Rivera finalizó el mural en Nueva York en 1971


── clip_067 · habla leída · 4 s
Lo que dicen en internet puede ser mentira o no


── clip_075 · habla leída · 7 s
El tiempo total de viaje es de 10 horas 35 minutos.


## 4 · Seudonimización

1. **Patrones automáticos:** DNI (8 dígitos), RUC (11), teléfonos móviles y correos → `[DNI]`, `[RUC]`, `[TELEFONO]`, `[CORREO]`.
2. **Nombres de persona:** el modelo NER `mrm8488/bert-spanish-cased-finetuned-ner` *sugiere* candidatos; la decisión es humana.
   - Personas privadas (entrevistadas en la calle, nombres mencionados de terceros) → seudónimo (`Persona_A`, …).
   - Figuras públicas o históricas en documentos públicos (Ricardo Palma, firmantes del Acta de 1821, autoridades en ejercicio en una resolución oficial) **se conservan**, porque no son datos personales privados.
3. Revisión final leyendo los textos: el procedimiento automático no sustituye la revisión humana.

In [8]:
import importlib; importlib.reload(H)
todos = pd.concat([ocr_res.set_index("id")["texto_ocr"], asr_res.set_index("id")["texto_asr"]])
candidatos = H.sugerir_personas(todos)
resumen = candidatos.groupby("nombre").agg(veces=("id", "size"), ids=("id", lambda s: ", ".join(sorted(set(s))))).sort_values("veces", ascending=False)
# Los nombres reales NO se muestran aquí porque el notebook se publica: se revisan en un archivo privado
resumen.to_csv(DATOS / "raw" / "candidatos_personas.csv")
print(f"{len(resumen)} nombres candidatos guardados en datos/raw/candidatos_personas.csv (privado, fuera de Git)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: mrm8488/bert-spanish-cased-finetuned-ner
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


112 nombres candidatos guardados en datos/raw/candidatos_personas.csv (privado, fuera de Git)


In [9]:
# Diccionario {nombre real: seudónimo} en un archivo privado (datos/raw/reemplazos.json, fuera de Git).
# Solo personas privadas; figuras públicas o históricas en documentos públicos se conservan.
ruta_reemplazos = DATOS / "raw" / "reemplazos.json"
REEMPLAZOS = json.loads(ruta_reemplazos.read_text(encoding="utf-8")) if ruta_reemplazos.exists() else {}
print(f"Personas privadas seudonimizadas: {len(REEMPLAZOS)}")

conteo_total = pd.Series(dtype=int)
for tabla, col in [(ocr_res, "texto_ocr"), (asr_res, "texto_asr")]:
    nuevos = []
    for texto in tabla[col]:
        limpio, c = H.seudonimizar_texto(texto, REEMPLAZOS)
        nuevos.append(limpio)
        conteo_total = conteo_total.add(pd.Series(c, dtype=int), fill_value=0)
    tabla[col] = nuevos

print(f"Entidades reemplazadas: {int(conteo_total.sum())}")
conteo_total.astype(int).rename("reemplazos").to_frame()

Personas privadas seudonimizadas: 9
Entidades reemplazadas: 18


,reemplazos
CORREO,0
DNI,1
NOMBRE,16
RUC,0
TELEFONO,1


## 5 · Resumen de descartes

In [10]:
print(f"Total descartados: {len(descartes)}")
descartes.fillna("").reset_index(drop=True)

Total descartados: 2


,id,archivo,motivo,archivo_audio
0,,,sin voz (<10 % de habla según VAD),datos/raw/audio/clips/fuente_06_01.wav
1,,,sin voz (<10 % de habla según VAD),datos/raw/audio/clips/fuente_12_00.wav


## 6 · Análisis exploratorio

Los títulos de los gráficos se calculan a partir de los datos, para que expresen la conclusión y no solo la variable.

In [11]:
ocr_res["caracteres_ocr"] = ocr_res["texto_ocr"].str.len()
asr_res["palabras_asr"] = asr_res["texto_asr"].str.replace(r"\[HABLANTE_\d+\]", "", regex=True).str.split().str.len()

conf = ocr_res.groupby("tipo")["confianza_media"].mean()
peor = conf.idxmin()
fig1 = px.histogram(ocr_res, x="tipo", color="tipo", category_orders={"tipo": ["impreso", "foto", "manuscrito"]},
                    title=f"El corpus está balanceado entre los tres tipos; los {peor}s tienen la menor confianza OCR ({conf[peor]:.2f})",
                    labels={"tipo": "Tipo de documento", "count": "Documentos"})
fig1.show()

In [12]:
por_fuente = asr_res.groupby("tipo_audio")["duracion_seg"].sum().div(60).sort_values(ascending=False)
fig2 = px.bar(asr_res.sort_values("duracion_seg", ascending=False), x="id", y="duracion_seg", color="tipo_audio",
              title=f"Las entrevistas aportan {por_fuente.get('entrevista', 0) / por_fuente.sum():.0%} de los "
                    f"{por_fuente.sum():.0f} minutos; las frases de OpenSLR son muchas pero cortas",
              labels={"duracion_seg": "Duración (s)", "id": "Clip", "tipo_audio": "Tipo de audio"})
fig2.update_xaxes(showticklabels=False)
fig2.show()

In [13]:
largos = pd.concat([
    ocr_res.assign(modalidad="OCR (caracteres)", longitud=ocr_res["caracteres_ocr"])[["modalidad", "longitud"]],
    asr_res.assign(modalidad="ASR (palabras)", longitud=asr_res["palabras_asr"])[["modalidad", "longitud"]],
])
med = largos.groupby("modalidad")["longitud"].median()
fig3 = px.histogram(largos, x="longitud", facet_col="modalidad", nbins=30, facet_col_spacing=0.08,
                    title=f"Las longitudes son muy desiguales: mediana de {med['OCR (caracteres)']:.0f} caracteres por documento "
                          f"y {med['ASR (palabras)']:.0f} palabras por clip",
                    labels={"longitud": "Longitud del texto extraído"})
fig3.update_xaxes(matches=None)
fig3.show()

## 7 · Validación: extremos de longitud

In [14]:
cols = ["id", "tipo", "caracteres_ocr", "texto_ocr"]
print("Documentos con OCR más largo"); display(ocr_res.nlargest(3, "caracteres_ocr")[cols])
print("Documentos con OCR más corto"); display(ocr_res.nsmallest(3, "caracteres_ocr")[cols])
cols = ["id", "tipo_audio", "duracion_seg", "palabras_asr", "texto_asr"]
print("Clips más largos"); display(asr_res.nlargest(3, "duracion_seg")[cols])
print("Clips más cortos"); display(asr_res.nsmallest(3, "duracion_seg")[cols])

Documentos con OCR más largo


,id,tipo,caracteres_ocr,texto_ocr
6,doc_007,impreso,9213,Bste periódieo so despacha en esta Capital\nLos avisos que deban samr en esto\nen a tienda del Señor Dorado calle de...
5,doc_006,impreso,5821,Eate periidion se doepaenn dn este Capltal\nLe ovive qoé deben spir an rd\nsa aates de lue custro de la sardo del di...
7,doc_008,impreso,5164,EL COMERCIO\n826\nras siendo el numero ultimo 183.\nS e está esperando un decreto del Gobierno\nuna chispa eléctrica...


Documentos con OCR más corto


,id,tipo,caracteres_ocr,texto_ocr
10,doc_011,foto,7,PICANTE
8,doc_009,foto,37,Baby Boy\nBARRIO\nCHINO\nCALLE\nCAPON\nlov
4,doc_005,impreso,71,NUM..1\nELPERUANO\nINDEPENDIENTE\nDEL SABADOISDE OCTUBRE DEIERS\nPROSPECTO.


Clips más largos


,id,tipo_audio,duracion_seg,palabras_asr,texto_asr
22,clip_023,entrevista,90.1,60,[HABLANTE_1] ¿Qué es el Perú para mí? Siento parte de la cultura pegana. [HABLANTE_2] ¿Cuáles son sus característica...
45,clip_046,entrevista,67.5,189,"Actualmente de hecho estoy desarrollando un proyecto pero todavía está en proceso de desarrollo, en una creación ini..."
16,clip_017,entrevista,66.6,142,"[HABLANTE_1] Toda la verdad, jóvenes, espero porque el futuro es de ustedes, nosotros ya nos vamos y tienen que defe..."


Clips más cortos


,id,tipo_audio,duracion_seg,palabras_asr,texto_asr
79,clip_080,habla leída,2.9,8,En su auto hay un botiquín de emergencia
88,clip_089,habla leída,3.3,6,Me interesa una bolsa color azul
66,clip_067,habla leída,3.8,10,Lo que dicen en internet puede ser mentira o no


**Coherencia con los histogramas:** _(completar: ¿los documentos más largos son impresos de página completa y los más cortos fotos de carteles? ¿Las palabras de cada clip son proporcionales a su duración, unas 2 a 3 palabras por segundo?)_

## 8 · Exportación

In [15]:
documentos = ocr_res.assign(modelo_ocr=MODELO_OCR, texto_referencia="")[
    ["id", "archivo", "tipo", "fuente", "texto_ocr", "modelo_ocr", "texto_referencia"]
    if "fuente" in ocr_res else ["id", "archivo", "tipo", "texto_ocr", "modelo_ocr", "texto_referencia"]]
if "fuente" not in documentos:
    documentos.insert(3, "fuente", ocr_res["fuente_url"].fillna("captura propia") + " (" + ocr_res["licencia"] + ")")

transcripciones = asr_res.assign(idioma="es (español peruano según la fuente)", modelo_asr=MODELO_ASR, texto_referencia="")[
    ["id", "archivo_audio", "duracion_seg", "idioma", "texto_asr", "modelo_asr", "n_hablantes", "texto_referencia",
     "fuente_id", "tipo_audio", "rtf"]]

openslr_fuente = pd.DataFrame([{"id": "openslr73", "modalidad": "audio", "fuente_url": "https://www.openslr.org/73/",
                                "licencia": "CC BY-SA 4.0", "autor": "Google (Guevara-Rukoz et al., 2020)",
                                "fecha_extraccion": FECHA, "acceso": "descargas.py bajar (datos.py de la Sesión 13, HTTP Range)"}])
fuentes_csv = pd.concat([fuentes, openslr_fuente])[["id", "modalidad", "fuente_url", "licencia", "acceso", "autor", "fecha_extraccion"]]

documentos.to_csv(DATOS / "documentos.csv", index=False)
transcripciones.to_csv(DATOS / "transcripciones.csv", index=False)
fuentes_csv.to_csv(DATOS / "fuentes.csv", index=False)

# Muestra ligera y publicable para el dashboard (datos/ejemplos/):
# - todas las imágenes (Commons, licencias libres, sin datos personales privados) a 900 px
# - audios solo de fuentes sin personas privadas: noticiarios de la BNP (dominio público) y OpenSLR 73 (CC BY-SA, con consentimiento)
from PIL import Image
EJ = DATOS / "ejemplos"; (EJ / "imagenes").mkdir(parents=True, exist_ok=True); (EJ / "audio").mkdir(parents=True, exist_ok=True)
for d in ocr_res.itertuples():
    img = Image.open(RAW_DOC / "prep" / f"{d.id}.jpg"); img.thumbnail((900, 900)); img.save(EJ / "imagenes" / f"{d.id}.jpg", quality=80)
publicables = pd.concat([transcripciones[transcripciones["tipo_audio"] == "noticiario"],
                         transcripciones[transcripciones["tipo_audio"] == "habla leída"].head(6)])
for c in publicables.itertuples():
    sf.write(EJ / "audio" / f"{c.id}.ogg", voz.cargar_audio(c.archivo_audio), voz.SR, format="OGG", subtype="VORBIS")
print(f"Ejemplos publicables: {len(ocr_res)} imágenes y {len(publicables)} audios")

for nombre, df in [("documentos", documentos), ("transcripciones", transcripciones), ("fuentes", fuentes_csv)]:
    print(nombre, df.shape)
display(documentos.head())
display(transcripciones.head())

Ejemplos publicables: 25 imágenes y 12 audios
documentos (25, 7)
transcripciones (97, 11)
fuentes (43, 7)


,id,archivo,tipo,fuente,texto_ocr,modelo_ocr,texto_referencia
0,doc_001,datos/raw/documentos/doc_001.jpg,impreso,https://commons.wikimedia.org/wiki/File:Acta_de_Creaci%C3%B3n_de_los_S%C3%ADmbolos_para_el_Distrito_de_Mache.jpg (CC...,SE RESUELVE:\n.- Artículo 1°- RECONOCER como lema ganador en el Concurso Público\nCreación del Lema para el Distrito...,"PaddleOCR 3.7.0 (lang=es, textline_orientation=on, sin unwarping, lado máx. 1600 px)",
1,doc_002,datos/raw/documentos/doc_002.jpg,impreso,https://commons.wikimedia.org/wiki/File:Pisco-punch-mercurio-peruano-1791.jpg (Public domain),Eol. 25\nMERCURIO PERUANO\nDEL DIA 13. DE ENERO DE 1791.\n\nEXAMEN HISTÓRICO DE LAS DIVERSIONES PÚBLICAS\nde las Nac...,"PaddleOCR 3.7.0 (lang=es, textline_orientation=on, sin unwarping, lado máx. 1600 px)",
2,doc_003,datos/raw/documentos/doc_003.jpg,impreso,https://commons.wikimedia.org/wiki/File:Mercurio_Peruano1.jpg (Public domain),ENES\nFol. r\n\nNum. I.\nや00\nMERCURIO PERUANO\nDEL DIA 2. DE ENERO DE 1791.\nIDEA CENERAL DEL PERÚ.\nque se anticip...,"PaddleOCR 3.7.0 (lang=es, textline_orientation=on, sin unwarping, lado máx. 1600 px)",
3,doc_004,datos/raw/documentos/doc_004.jpg,impreso,https://commons.wikimedia.org/wiki/File:Constituci%C3%B3n_Pol%C3%ADtica_de_la_Rep%C3%BAblica_Peruana_de_1823.jpg (Pu...,NSTTY\nOLTICA\nV1 3A\n\nPEUUNA\nJURAANLIMA\nEL 20 DE NOVIEMBRE\nDE 1823.\nLIMA: 1825.\nAPRENTADEL ESTADO POR JGONZALE,"PaddleOCR 3.7.0 (lang=es, textline_orientation=on, sin unwarping, lado máx. 1600 px)",
4,doc_005,datos/raw/documentos/doc_005.jpg,impreso,https://commons.wikimedia.org/wiki/File:Primera_edici%C3%B3n_de_El_Peruano_(1825).jpg (Public domain),NUM..1\nELPERUANO\nINDEPENDIENTE\nDEL SABADOISDE OCTUBRE DEIERS\nPROSPECTO.,"PaddleOCR 3.7.0 (lang=es, textline_orientation=on, sin unwarping, lado máx. 1600 px)",


,id,archivo_audio,duracion_seg,idioma,texto_asr,modelo_asr,n_hablantes,texto_referencia,fuente_id,tipo_audio,rtf
0,clip_001,datos/raw/audio/clips/fuente_01_00.wav,46.3,es (español peruano según la fuente),"[HABLANTE_1] Buenos días, aquí son las 10 y 28 y estoy con una señora que va a darme su opinión acerca del partido d...","faster-whisper large-v3-turbo (int8, CPU, VAD on, beam 5)",2,,fuente_01,entrevista,0.64
1,clip_002,datos/raw/audio/clips/fuente_01_01.wav,45.5,es (español peruano según la fuente),[HABLANTE_1] En este partido [HABLANTE_2] Muy bien [HABLANTE_1] Y la tercera pregunta es acerca de ¿Cuál es el resul...,"faster-whisper large-v3-turbo (int8, CPU, VAD on, beam 5)",2,,fuente_01,entrevista,0.33
2,clip_003,datos/raw/audio/clips/fuente_01_02.wav,44.5,es (español peruano según la fuente),"[HABLANTE_1] ¿Qué piensa del fútbol como unión de masas? Porque como puede ver, muy pronto en muchos locales va a ha...","faster-whisper large-v3-turbo (int8, CPU, VAD on, beam 5)",2,,fuente_01,entrevista,0.28
3,clip_004,datos/raw/audio/clips/fuente_02_00.wav,47.9,es (español peruano según la fuente),"[HABLANTE_1] Buenos días profesores y compañeros, hoy estoy cerca de aquí por mi casa y voy a hacer una entrevista a...","faster-whisper large-v3-turbo (int8, CPU, VAD on, beam 5)",3,,fuente_02,entrevista,0.26
4,clip_005,datos/raw/audio/clips/fuente_02_01.wav,47.2,es (español peruano según la fuente),al mundial puede decir su nombre Persona_B bien sigo con la siguiente pregunta qué jugador piensa que va a ser más d...,"faster-whisper large-v3-turbo (int8, CPU, VAD on, beam 5)",1,,fuente_02,entrevista,0.25
